#  ALAZ — TLE & Space Weather Data Pipeline
**Project:** ALAZ Solar Alert — Real-time Turkish Satellite Conjunction & Solar Impact Monitor  
**Team:** Karaman | TUA Astro Hackathon 2026  

This notebook builds the **master training dataset** by:
1. Normalizing and merging raw TLE archives (2025 ZIP + 2026 GP_History API)
2. Extracting the 39 Turkish satellite assets (active and historical)
3. Parsing orbital parameters (B*, Mean Motion, Epoch) per the SGP4/TLE specification
4. Fusing satellite records with high-resolution NOAA space weather indices (Kp, F10.7, ISN)

**Data Sources:**
- TLE: Space-Track.org — annual ZIP archive (2025) + GP_History API (2026 Q1)
- Space Weather: NOAA SWPC — SW-Last5Years.csv (observed records only)


## 1. Environment Configuration
Dynamic directory setup ensures portability across machines.
All raw and processed data paths are resolved relative to the project root (`astro/`).


In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# --- Detect project root regardless of notebook run location ---
current_path = os.getcwd()
BASE_DIR = os.path.abspath(os.path.join(current_path, "..")) if "notebooks" in current_path else current_path

RAW_DIR       = os.path.join(BASE_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

# --- Source file paths ---
tle_2025_src  = os.path.join(RAW_DIR, "tle2025.txt")
tle_2026_src  = os.path.join(RAW_DIR, "tle2026_q1.txt")
master_tle_path = os.path.join(PROCESSED_DIR, "ALAZ_Master_TLE_2025_2026.txt")

print(f"Project root : {BASE_DIR}")
print(f"Raw data     : {RAW_DIR}")
print(f"Processed    : {PROCESSED_DIR}")


Project root : C:\Users\DELL\Desktop\astro
Raw data     : C:\Users\DELL\Desktop\astro\data\raw
Processed    : C:\Users\DELL\Desktop\astro\data\processed


## 2. Source Data Integrity Fix — tle2026_q1.txt
The GP_History API occasionally returns duplicate TLE pairs within the same response payload.
This cell performs a **one-time deduplication** of the 2026 source file before the main pipeline runs.
It is safe to re-run: if no duplicates exist, the file is written back unchanged.


In [2]:
def deduplicate_tle_file(file_path):
    """
    Reads a 2LE TLE file, removes exact duplicate (Line1, Line2) pairs,
    and overwrites the file with only unique pairs.
    """
    if not os.path.exists(file_path):
        print(f"  SKIP: {file_path} not found.")
        return

    with open(file_path, 'r') as f:
        lines = [l for l in f if l.strip()]

    clean_lines = []
    seen = set()
    for i in range(0, len(lines) - 1, 2):
        l1 = lines[i].rstrip()
        l2 = lines[i + 1].rstrip()
        key = (l1, l2)
        if key not in seen:
            seen.add(key)
            clean_lines.extend([l1, l2])

    original_pairs = len(lines) // 2
    clean_pairs    = len(clean_lines) // 2
    removed        = original_pairs - clean_pairs

    with open(file_path, 'w') as f:
        for line in clean_lines:
            f.write(line + '\n')

    print(f"  {os.path.basename(file_path)}: {original_pairs:,} pairs -> {clean_pairs:,} unique pairs ({removed} duplicates removed)")


print("Running source file deduplication...")
deduplicate_tle_file(tle_2026_src)
print("Done.")


Running source file deduplication...
  tle2026_q1.txt: 9,919 pairs -> 9,919 unique pairs (0 duplicates removed)
Done.


## 3. Raw Data Ingestion & Format-Aware Integrity Validation
Space-Track provides TLE data in two formats depending on the download method:
- **3LE** (Name + Line 1 + Line 2): annual ZIP cloud archives
- **2LE** (Line 1 + Line 2 only): GP_History API with `format/tle`

This pipeline **auto-detects** the format of each source file and normalizes both into a
uniform 2LE master, preventing the most common cause of TLE parse misalignment.
Only structurally valid pairs (Line 1 starting with `1 `, Line 2 with `2 `) are written.


In [3]:
def detect_tle_format(file_path):
    """
    Detects whether a TLE file is 2LE or 3LE by inspecting the first non-empty line.
    A TLE Line 1 always starts with '1 '; a name line does not.
    Returns 'tle2' or 'tle3'.
    """
    with open(file_path, 'r') as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            return 'tle2' if (line.startswith('1 ') or line.startswith('2 ')) else 'tle3'
    return 'tle2'


def normalize_to_2le(file_path):
    """
    Reads a TLE file (2LE or 3LE) and yields validated (line1, line2) string pairs.
    Name lines in 3LE files are discarded; output is always uniform 2LE.
    """
    fmt = detect_tle_format(file_path)
    print(f"  Format detected: {fmt.upper()} — {os.path.basename(file_path)}")

    with open(file_path, 'r') as f:
        lines = [l.rstrip('\n') for l in f if l.strip()]

    step = 3 if fmt == 'tle3' else 2
    malformed = 0

    for i in range(0, len(lines) - (step - 1), step):
        l1 = lines[i + 1] if fmt == 'tle3' else lines[i]
        l2 = lines[i + 2] if fmt == 'tle3' else lines[i + 1]

        if l1.startswith('1 ') and l2.startswith('2 '):
            yield l1, l2
        else:
            malformed += 1

    if malformed:
        print(f"  WARNING: {malformed} malformed pairs skipped in {os.path.basename(file_path)}")


def merge_and_validate_raw_tle(source_list, output_file):
    """
    Merges TLE source files of any format (2LE or 3LE) into a single
    normalized 2LE master file. Only validated pairs are written.
    """
    print(f"Starting normalized merge -> {os.path.basename(output_file)}")
    total_pairs = 0

    with open(output_file, 'w') as out:
        for file_path in source_list:
            if not os.path.exists(file_path):
                print(f"  CRITICAL ERROR: {file_path} not found — skipping.")
                continue
            print(f"  Merging: {os.path.basename(file_path)}")
            for l1, l2 in normalize_to_2le(file_path):
                out.write(l1 + '\n')
                out.write(l2 + '\n')
                total_pairs += 1

    print(f"\n  Merge complete: {total_pairs:,} validated TLE pairs written.")
    print(f"  Master file  : {output_file}")
    return total_pairs


merge_and_validate_raw_tle([tle_2025_src, tle_2026_src], master_tle_path)


Starting normalized merge -> ALAZ_Master_TLE_2025_2026.txt
  Merging: tle2025.txt
  Format detected: TLE2 — tle2025.txt
  Merging: tle2026_q1.txt
  Format detected: TLE2 — tle2026_q1.txt

  Merge complete: 20,789,520 validated TLE pairs written.
  Master file  : C:\Users\DELL\Desktop\astro\data\processed\ALAZ_Master_TLE_2025_2026.txt


20789520

## 4. Targeted Satellite Asset Extraction
Scanning the 41M-line master TLE for the **39 active Turkish satellite assets**.
NORAD IDs are stored in a Python `set` for O(1) membership testing.
Every pair is structurally validated before the NORAD ID lookup.

**TLE Line 1 column reference (0-indexed):**
- `[0]` = Line number (`'1'`)
- `[2:7]` = NORAD Catalog Number (right-justified, zero/space padded)


In [4]:
# 39 active Turkish satellite NORAD IDs (set for O(1) lookup)
TARGET_NORAD_IDS = {
    "23200", "23949", "26666", "27943", "33056", "35935", "37791", "39030",
    "39152", "39522", "40984", "41875", "47306", "50212", "55012", "56178",
    "60233", "60472", "60475", "60522", "60524", "62653", "62695", "62703",
    "62709", "62715", "64534", "64553", "64555", "64557", "66294", "66297",
    "66298", "66299", "66777", "67390", "67400", "67401", "67402"
}

filtered_tle_out = os.path.join(PROCESSED_DIR, "ALAZ_Filtered_Turkish_Assets.txt")


def extract_specific_assets(master_in, filtered_out, id_set):
    """
    Reads the normalized 2LE master file and extracts only pairs whose
    NORAD ID (Line 1, columns [2:7]) is in the target set.
    """
    print(f"Scanning master TLE for {len(id_set)} target assets...")
    matches  = 0
    skipped  = 0

    with open(master_in, 'r') as infile, open(filtered_out, 'w') as outfile:
        lines = [l.rstrip('\n') for l in infile if l.strip()]

        for i in range(0, len(lines) - 1, 2):
            l1 = lines[i]
            l2 = lines[i + 1]

            if not (l1.startswith('1 ') and l2.startswith('2 ')):
                skipped += 1
                continue

            norad = l1[2:7].strip()
            if norad in id_set:
                outfile.write(l1 + '\n')
                outfile.write(l2 + '\n')
                matches += 1

    print(f"\n  Extraction complete.")
    print(f"  Matched records  : {matches:,}")
    print(f"  Malformed skipped: {skipped}")
    print(f"  Output file      : {filtered_out}")
    return matches


extract_specific_assets(master_tle_path, filtered_tle_out, TARGET_NORAD_IDS)


Scanning master TLE for 39 target assets...

  Extraction complete.
  Matched records  : 35,882
  Malformed skipped: 0
  Output file      : C:\Users\DELL\Desktop\astro\data\processed\ALAZ_Filtered_Turkish_Assets.txt


35882

## 5. TLE Parameter Parsing & Feature Engineering
Raw orbital strings are transformed into a structured time-series DataFrame.

**Key parameters extracted (per SGP4/TLE specification):**

| Parameter | TLE Location | Notes |
|---|---|---|
| NORAD ID | Line 1, cols 3–7 (`[2:7]`) | Right-justified, zero-padded |
| Epoch | Line 1, cols 19–32 (`[18:32]`) | YYDDD.DDDDDDDD; YY≥57→19YY, YY<57→20YY |
| B* Drag Term | Line 1, cols 54–61 (`[53:61]`) | Decimal assumed; ±MMMMM±E shorthand |
| Mean Motion | Line 2, cols 53–63 (`[52:63]`) | Revolutions/day |

**Year disambiguation rule** (Space-Track specification):
- `YY >= 57` → `1900 + YY` (e.g. `98` → `1998`) — covers TURKSAT 1B/1C/2A
- `YY < 57`  → `2000 + YY` (e.g. `25` → `2025`)

**Note on negative B* values:** Physically, B* should be non-negative.
Negative values in Space-Track data reflect orbit-fit residuals and are a known
data characteristic, not a pipeline error.


In [5]:
# Satellite name mapping: NORAD ID -> human-readable name
SAT_NAME_MAP = {
    "23200": "TURKSAT 1B",      "23949": "TURKSAT 1C",      "26666": "TURKSAT 2A",
    "27943": "BILSAT 1",        "33056": "TURKSAT 3A",      "35935": "ITUPSAT 1",
    "37791": "RASAT",           "39030": "GOKTURK 2",       "39152": "TURKSAT 3U",
    "39522": "TURKSAT 4A",      "40984": "TURKSAT 4B",      "41875": "GOKTURK-1A",
    "47306": "TURKSAT 5A",      "50212": "TURKSAT 5B",      "55012": "CONNECTA T1.2",
    "56178": "IMECE",           "60233": "TURKSAT 6A",      "60472": "CONNECTA IOT-1",
    "60475": "CONNECTA IOT-3",  "60522": "CONNECTA IOT-2",  "60524": "CONNECTA IOT-4",
    "62653": "PAUSAT-1",        "62695": "CONNECTA IOT-8",  "62703": "CONNECTA IOT-5",
    "62709": "CONNECTA IOT-6",  "62715": "CONNECTA IOT-7",  "64534": "CONNECTA IOT-9",
    "64553": "CONNECTA IOT-12", "64555": "CONNECTA IOT-10", "64557": "CONNECTA IOT-11",
    "66294": "SEMI-1L",         "66297": "SEMI-1N",         "66298": "SEMI-1P",
    "66299": "FGN-100-D2",      "66777": "LUNA-1",          "67390": "CONNECTA IOT-16",
    "67400": "CONNECTA IOT-13", "67401": "CONNECTA IOT-14", "67402": "CONNECTA IOT-15"
}


def parse_epoch(epoch_str):
    """
    Converts a TLE epoch string (YYDDD.DDDDDDDD) to a tz-naive pandas Timestamp.

    Year rule per Space-Track documentation:
      YY >= 57  ->  1900 + YY
      YY <  57  ->  2000 + YY
    """
    epoch_str = epoch_str.strip()
    yy = int(epoch_str[:2])
    year = 1900 + yy if yy >= 57 else 2000 + yy
    day_of_year = float(epoch_str[2:])
    base = pd.Timestamp(year=year, month=1, day=1)
    return base + pd.to_timedelta(day_of_year - 1.0, unit='D')


def parse_bstar(b_str):
    """
    Parses TLE B* drag term from Line 1 columns [53:61].

    Examples:
      ' 16538-3' ->  1.6538e-4
      '-11234-1' -> -1.1234e-2
      ' 00000-0' ->  0.0
    """
    try:
        raw = str(b_str)
        if len(raw) < 8:
            raw = raw.rjust(8)

        field = raw[:8]

        mantissa_sign = -1.0 if field[0] == '-' else 1.0
        mantissa_digits = field[1:6]
        exponent_str = field[6:8]

        if not mantissa_digits.strip():
            return 0.0

        mantissa = mantissa_sign * float(f"0.{mantissa_digits}")
        exponent = int(exponent_str)

        return mantissa * (10 ** exponent)

    except Exception:
        return 0.0


def build_alaz_dataframe(file_path, name_mapping):
    """
    Reads the validated 2LE filtered TLE file and builds a structured DataFrame.
    Applies deduplication on (NORAD_ID, EPOCH).
    """
    print("Parsing TLE strings into structured DataFrame...")
    data_rows = []
    parse_errors = 0

    with open(file_path, 'r') as f:
        lines = [l.rstrip('\n') for l in f if l.strip()]

    for i in range(0, len(lines) - 1, 2):
        l1 = lines[i]
        l2 = lines[i + 1]

        if not (l1.startswith('1 ') and l2.startswith('2 ')):
            parse_errors += 1
            continue

        try:
            norad_id = l1[2:7].strip()
            ts = parse_epoch(l1[18:32])
            bstar = parse_bstar(l1[53:61])
            mean_motion = float(l2[52:63])

            data_rows.append({
                "SAT_NAME": name_mapping.get(norad_id, "UNKNOWN"),
                "NORAD_ID": norad_id,
                "EPOCH": ts,
                "DATE": ts.date(),
                "BSTAR": bstar,
                "MEAN_MOTION": mean_motion,
            })
        except Exception as e:
            parse_errors += 1
            print(f"  WARNING: Parse error at pair {i // 2}: {e}")

    df = pd.DataFrame(data_rows)
    df = df.sort_values(by=["NORAD_ID", "EPOCH"]).reset_index(drop=True)

    before = len(df)
    df = df.drop_duplicates(subset=["NORAD_ID", "EPOCH"]).reset_index(drop=True)
    removed = before - len(df)

    print(f"\n  Parsing complete.")
    print(f"  Records parsed       : {before:,}")
    print(f"  Duplicates removed   : {removed}")
    print(f"  Final records        : {len(df):,}")
    print(f"  Satellites covered   : {df['SAT_NAME'].nunique()}")
    print(f"  Date range           : {df['EPOCH'].min()} -> {df['EPOCH'].max()}")
    print(f"  Parse errors         : {parse_errors}")
    return df


df_alaz = build_alaz_dataframe(filtered_tle_out, SAT_NAME_MAP)

Parsing TLE strings into structured DataFrame...

  Parsing complete.
  Records parsed       : 35,882
  Duplicates removed   : 3242
  Final records        : 32,640
  Satellites covered   : 39
  Date range           : 2025-01-01 00:01:28.758720 -> 2026-03-28 14:32:26.845728
  Parse errors         : 0


## 6. High-Resolution Space Weather Fusion
Each satellite record is aligned with its corresponding **3-hourly NOAA Kp window**
and daily solar indices (F10.7, ISN).

**Merge strategy:**
- `DATE` cast to `datetime.date` explicitly on both sides — prevents silent type-mismatch
- `PERIOD = (EPOCH.hour // 3) + 1` maps UTC hour 0–23 to Kp windows 1–8 (matching KP1–KP8 columns)
- Only **observed** SW records (`F10.7_DATA_TYPE == 'OBS'`) are used — forecast/predicted rows excluded
- Date range restricted to project window: 2025-01-01 → 2026-03-29
- Pre-merge overlap diagnostic raises `ValueError` if date ranges do not intersect
- Post-merge null-check reports SW coverage quality

**Note on NaN values:** A small number of records (~0.4%) may have NaN SW values
due to NOAA publication delays for the most recent dates. This does not affect model training.


In [6]:
def integrate_space_weather(df_satellite, sw_filename):
    """
    Fuses satellite telemetry (TLE-derived) with 3-hourly NOAA space weather indices.

    Merge keys:
      DATE   - datetime.date (explicit cast on both sides)
      PERIOD - int 1-8 representing 3-hour UTC window

    Join: LEFT - every satellite record is preserved;
    SW rows without a matching date/period receive NaN.
    """
    sw_path = os.path.join(RAW_DIR, sw_filename)
    if not os.path.exists(sw_path):
        raise FileNotFoundError(f"Space weather file not found: {sw_path}")

    print(f"Loading space weather data: {sw_filename}")
    df_sw = pd.read_csv(sw_path)

    # --- Keep only observed records; exclude forecasts (PRD, PRM, etc.) ---
    df_sw = df_sw[df_sw['F10.7_DATA_TYPE'] == 'OBS'].copy()

    # --- Cast DATE to datetime.date ---
    df_sw['DATE'] = pd.to_datetime(df_sw['DATE'], errors='coerce').dt.date
    bad = df_sw['DATE'].isna().sum()
    if bad:
        print(f"  WARNING: {bad} SW rows had unparseable DATE — dropped.")
    df_sw = df_sw.dropna(subset=['DATE'])

    # --- Restrict to project date range ---
    START_DATE = pd.Timestamp('2025-01-01').date()
    END_DATE   = pd.Timestamp('2026-03-29').date()
    df_sw = df_sw[
        (df_sw['DATE'] >= START_DATE) &
        (df_sw['DATE'] <= END_DATE)
    ].reset_index(drop=True)
    print(f"  SW filtered : {len(df_sw)} observed days "
          f"({df_sw['DATE'].min()} -> {df_sw['DATE'].max()})")

    # --- Reshape KP1..KP8 into long format (one row per date-period) ---
    kp_cols = [f'KP{i}' for i in range(1, 9)]
    missing = [c for c in kp_cols if c not in df_sw.columns]
    if missing:
        raise ValueError(f"Missing Kp columns in SW file: {missing}")

    df_sw_3h = df_sw.melt(
        id_vars=['DATE'], value_vars=kp_cols,
        var_name='PERIOD', value_name='KP_INDEX'
    )
    df_sw_3h['PERIOD'] = df_sw_3h['PERIOD'].str.replace('KP', '', regex=False).astype(int)

    # --- Attach daily indices (F10.7, ISN) to the 3-hour frame ---
    daily_cols = [c for c in ['DATE', 'ISN', 'F10.7_OBS', 'F10.7_ADJ'] if c in df_sw.columns]
    if len(daily_cols) > 1:
        df_sw_3h = pd.merge(
            df_sw_3h,
            df_sw[daily_cols].drop_duplicates('DATE'),
            on='DATE', how='left'
        )

    # --- Satellite side: assign PERIOD + re-cast DATE defensively ---
    df_sat = df_satellite.copy()
    df_sat['PERIOD'] = (df_sat['EPOCH'].dt.hour // 3) + 1
    df_sat['DATE']   = pd.to_datetime(df_sat['DATE']).dt.date

    # --- Pre-merge overlap diagnostic ---
    sat_dates = set(df_sat['DATE'].unique())
    sw_dates  = set(df_sw_3h['DATE'].unique())
    overlap   = sat_dates & sw_dates
    print(f"\n  Satellite date range : {min(sat_dates)} -> {max(sat_dates)}")
    print(f"  SW file date range   : {min(sw_dates)} -> {max(sw_dates)}")
    print(f"  Overlapping dates    : {len(overlap)} of {len(sat_dates)} satellite dates")

    if len(overlap) == 0:
        raise ValueError(
            "MERGE WOULD PRODUCE ZERO MATCHES: DATE ranges do not overlap. "
            "Check SW file coverage or DATE column format."
        )

    # --- Final left join ---
    df_master = pd.merge(df_sat, df_sw_3h, on=['DATE', 'PERIOD'], how='left')

    null_kp  = df_master['KP_INDEX'].isna().sum()
    null_pct = 100 * null_kp / len(df_master)
    if null_pct > 10:
        print(f"  WARNING: {null_pct:.1f}% of records have no SW match.")
    else:
        print(f"  SW fusion quality: {100 - null_pct:.1f}% of records matched.")

    # --- Save master dataset ---
    output_path = os.path.join(PROCESSED_DIR, "ALAZ_HighRes_Training_Data.csv")
    df_master.to_csv(output_path, index=False)

    print("\n" + "-" * 55)
    print(f"  ALAZ MASTER DATASET SAVED: {output_path}")
    print(f"  Final shape: {df_master.shape[0]:,} rows x {df_master.shape[1]} columns")
    return df_master


df_master = integrate_space_weather(df_alaz, "SW-Last5Years.csv")
print(df_master.head(10).to_string())


Loading space weather data: SW-Last5Years.csv
  SW filtered : 450 observed days (2025-01-01 -> 2026-03-28)

  Satellite date range : 2025-01-01 -> 2026-03-28
  SW file date range   : 2025-01-01 -> 2026-03-28
  Overlapping dates    : 447 of 449 satellite dates
  SW fusion quality: 99.6% of records matched.

-------------------------------------------------------
  ALAZ MASTER DATASET SAVED: C:\Users\DELL\Desktop\astro\data\processed\ALAZ_HighRes_Training_Data.csv
  Final shape: 32,640 rows x 11 columns
     SAT_NAME NORAD_ID                      EPOCH        DATE  BSTAR  MEAN_MOTION  PERIOD  KP_INDEX    ISN  F10.7_OBS  F10.7_ADJ
0  TURKSAT 1B    23200 2025-01-01 03:34:01.347744  2025-01-01    0.0     0.990458       2      53.0  198.0      219.2      211.9
1  TURKSAT 1B    23200 2025-01-01 15:33:46.410048  2025-01-01    0.0     0.990460       6      80.0  198.0      219.2      211.9
2  TURKSAT 1B    23200 2025-01-02 03:33:23.222880  2025-01-02    0.0     0.990462       2      20.0  187.0

## 7. Dataset Validation
Final sanity checks on the master dataset before passing to EDA and modelling.


In [7]:
print("=" * 55)
print("ALAZ MASTER DATASET — VALIDATION REPORT")
print("=" * 55)

# --- Schema ---
print("\n[1] Schema & null counts")
print(df_master.dtypes)
print()
print(df_master.isnull().sum())

# --- Duplicates ---
dups = df_master.duplicated(subset=['NORAD_ID', 'EPOCH']).sum()
print(f"\n[2] Duplicate (NORAD_ID, EPOCH) rows: {dups}")

# --- Date range ---
print(f"\n[3] Epoch range")
print(f"    Min: {df_master['EPOCH'].min()}")
print(f"    Max: {df_master['EPOCH'].max()}")

# --- Records per satellite ---
print("\n[4] Records per satellite")
print(df_master.groupby('SAT_NAME')['EPOCH'].count().sort_values().to_string())

# --- KP_INDEX distribution ---
print("\n[5] KP_INDEX distribution")
print(df_master['KP_INDEX'].describe())

# --- BSTAR by satellite (physical consistency check) ---
print("\n[6] BSTAR summary by satellite (GEO expected near 0.0)")
bstar = df_master.groupby('SAT_NAME')['BSTAR'].agg(['mean', 'min', 'max'])
print(bstar.to_string())


ALAZ MASTER DATASET — VALIDATION REPORT

[1] Schema & null counts
SAT_NAME                  str
NORAD_ID                  str
EPOCH          datetime64[ns]
DATE                   object
BSTAR                 float64
MEAN_MOTION           float64
PERIOD                  int32
KP_INDEX              float64
ISN                   float64
F10.7_OBS             float64
F10.7_ADJ             float64
dtype: object

SAT_NAME         0
NORAD_ID         0
EPOCH            0
DATE             0
BSTAR            0
MEAN_MOTION      0
PERIOD           0
KP_INDEX       131
ISN            131
F10.7_OBS      131
F10.7_ADJ      131
dtype: int64

[2] Duplicate (NORAD_ID, EPOCH) rows: 0

[3] Epoch range
    Min: 2025-01-01 00:01:28.758720
    Max: 2026-03-28 14:32:26.845728

[4] Records per satellite
SAT_NAME
SEMI-1P             191
CONNECTA IOT-14     195
CONNECTA IOT-13     208
CONNECTA IOT-15     213
CONNECTA IOT-16     214
SEMI-1N             215
SEMI-1L             223
FGN-100-D2          233
LUNA-1   

In [8]:
# Sort chronologically
df_master = df_master.sort_values(by=['EPOCH', 'NORAD_ID']).reset_index(drop=True)

# Handle missing space weather data safely
# Since KP / ISN / F10.7 are global time-based variables, fill only within identical DATE-PERIOD groups
sw_cols = ['KP_INDEX', 'ISN', 'F10.7_OBS', 'F10.7_ADJ']

for col in sw_cols:
    df_master[col] = df_master.groupby(['DATE', 'PERIOD'])[col].transform(
        lambda x: x.ffill().bfill()
    )

# Final safety check
print("Remaining null counts in SW columns:")
print(df_master[sw_cols].isnull().sum())

# Export final cleaned dataset to CSV
csv_path = os.path.join(PROCESSED_DIR, "ALAZ_HighRes_Training_Data.csv")
df_master.to_csv(csv_path, index=False)

print("Success! Dataset fully processed and saved as CSV.")
print(f"Saved to: {csv_path}")
print(f"Final shape: {df_master.shape}")

Remaining null counts in SW columns:
KP_INDEX     131
ISN          131
F10.7_OBS    131
F10.7_ADJ    131
dtype: int64
Success! Dataset fully processed and saved as CSV.
Saved to: C:\Users\DELL\Desktop\astro\data\processed\ALAZ_HighRes_Training_Data.csv
Final shape: (32640, 11)
